# Stage 6 — Inference на произвольном mp4

Пайплайн:
1. yt-dlp качает видео (≤60 с), сэмплируем 1 кадр/сек
2. Прогоняем каждый кадр через Inception-v3 + YT8M PCA → **1024-мерный** вектор на кадр
3. Аудио: в `feature_extractor.extract_audio_features` реального VGGish нет — отдаёт нули. Для наших моделей это нормально (в stage 3 мы тренировались с `drop_audio_p=0.1`, модель умеет работать без аудио)
4. Получаем `(T, 1024)` rgb + `(T, 128)` audio, падим/обрезаем до `L`, нормализуем теми же `norm_stats.npz` из train, прогоняем через модель

В отличие от старого inference здесь **не усредняем** кадры — передаём в модель как последовательность.

In [1]:
# ============================================================
# 6.0 — Импорты и конфиг
# ============================================================
import sys, json, subprocess
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence
import cv2
import warnings; warnings.filterwarnings('ignore')

PROJECT_DIR = Path('..').resolve()
sys.path.insert(0, str(PROJECT_DIR))
import feature_extractor as yt8m_fe

SRC_DIR    = Path('../data2/frame_processed')
STAGE3_DIR = Path('../data2/frame_stage3')
STAGE5_DIR = Path('../data2/frame_stage5')
INFER_DIR  = Path('../data2/frame_inference'); INFER_DIR.mkdir(exist_ok=True)

cfg_src = json.load(open(SRC_DIR    / 'config.json'))
cfg     = json.load(open(STAGE3_DIR / 'config.json'))
L         = cfg['L']
DIM_RGB   = cfg['dim_rgb']
DIM_AUD   = cfg['dim_audio']
N_CLASSES = cfg['n_classes']
GENRES    = cfg['genres']
DEQ_S     = cfg_src['dequant_scale']
DEQ_B     = cfg_src['dequant_bias']
norm      = np.load(STAGE3_DIR / 'norm_stats.npz')
DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'L={L}  GENRES={GENRES}')

I0000 00:00:1776841776.565338   20489 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1776841776.600018   20489 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1776841777.366262   20489 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


L=60  GENRES=['Animals', 'Animation', 'Beauty', 'Dance', 'Film', 'Food', 'Gaming', 'Music', 'Performance', 'Sports', 'Tech', 'Vehicles']


In [2]:
# ============================================================
# 6.1 — Повтор определений моделей (не зависим от других notebooks)
# ============================================================
class FlattenMLP(nn.Module):
    def __init__(self, n_frames, dim_rgb, dim_aud, n_classes,
                 hidden=(1024, 512), dropout=0.4):
        super().__init__()
        self.n_frames = n_frames
        in_dim = n_frames * (dim_rgb + dim_aud)
        layers = []; prev = in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h),
                       nn.GELU(), nn.Dropout(dropout)]; prev = h
        layers.append(nn.Linear(prev, n_classes))
        self.net = nn.Sequential(*layers)
    def forward(self, rgb, aud, lengths=None):
        K = self.n_frames
        x = torch.cat([rgb[:, :K], aud[:, :K]], dim=-1).reshape(rgb.size(0), -1)
        return self.net(x)

class FrameRNN(nn.Module):
    def __init__(self, dim_rgb, dim_aud, n_classes,
                 proj_dim=256, rnn_hidden=256, rnn_layers=1,
                 bidirectional=True, dropout=0.3, cell='gru'):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(dim_rgb+dim_aud, proj_dim),
                                  nn.LayerNorm(proj_dim), nn.GELU(), nn.Dropout(dropout))
        RNN = {'gru': nn.GRU, 'lstm': nn.LSTM}[cell]
        self.rnn = RNN(proj_dim, rnn_hidden, num_layers=rnn_layers,
                       batch_first=True, bidirectional=bidirectional,
                       dropout=dropout if rnn_layers > 1 else 0.0)
        out_dim = rnn_hidden * (2 if bidirectional else 1)
        self.head = nn.Sequential(nn.LayerNorm(out_dim), nn.Dropout(dropout),
                                  nn.Linear(out_dim, n_classes))
        self.bi = bidirectional; self.cell = cell
    def forward(self, rgb, aud, lengths):
        x = self.proj(torch.cat([rgb, aud], dim=-1))
        lens_cpu = lengths.detach().cpu().clamp(min=1)
        packed = pack_padded_sequence(x, lens_cpu, batch_first=True, enforce_sorted=False)
        _, h = self.rnn(packed)
        if self.cell == 'lstm': h = h[0]
        h = torch.cat([h[-2], h[-1]], dim=-1) if self.bi else h[-1]
        return self.head(h)

def load_model(kind):
    if kind == 'flatten':
        m = FlattenMLP(min(30, L), DIM_RGB, DIM_AUD, N_CLASSES,
                       hidden=(1024, 512), dropout=0.4)
    else:
        m = FrameRNN(DIM_RGB, DIM_AUD, N_CLASSES,
                     proj_dim=256, rnn_hidden=256, bidirectional=True,
                     dropout=0.3, cell='gru')
    m.load_state_dict(torch.load(STAGE5_DIR / f'best_{kind}.pt', weights_only=True))
    return m.to(DEVICE).eval()

In [3]:
# ============================================================
# 6.2 — Инициализация YT8M-экстрактора (Inception + PCA)
# ============================================================
print('init YT8M feature extractor (первый раз скачает ~100MB)...')
extractor = yt8m_fe.YouTube8MFeatureExtractor()
print('ok')

init YT8M feature extractor (первый раз скачает ~100MB)...
ok


W0000 00:00:1776841777.989103   20489 op_def_util.cc:371] Op BatchNormWithGlobalNormalization is deprecated. It will cease to work in GraphDef version 9. Use tf.nn.batch_normalization().
W0000 00:00:1776841778.081096   20489 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
W0000 00:00:1776841778.084424   20489 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
I0000 00:00:1776841778.166528   20489 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13473 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5060 Ti, pci bus id: 0000:01:00.0, compute capability: 12.0a


In [4]:
# ============================================================
# 6.3 — Per-frame экстрактор признаков
# ============================================================
def extract_frame_features(video_path, fps_sample=1, max_frames=None):
    """Возвращает rgb[T,1024] + audio[T,128] где T — число сэмплированных кадров.
    Кадры YT8M-совместимые (Inception + PCA).
    Аудио: нули (VGGish не подключён) — модели умеют работать без звука.
    """
    if max_frames is None:
        max_frames = L
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    step = max(int(round(fps / fps_sample)), 1)

    rgb_feats = []
    frame_idx = 0
    while True:
        ok, frame = cap.read()
        if not ok: break
        if frame_idx % step == 0:
            # BGR → RGB
            frame_rgb = frame[:, :, ::-1]
            feat = extractor.extract_rgb_frame_features(frame_rgb, apply_pca=True)
            rgb_feats.append(feat.astype(np.float32))
            if len(rgb_feats) >= max_frames: break
        frame_idx += 1
    cap.release()
    if not rgb_feats:
        raise RuntimeError(f'Не удалось извлечь кадры из {video_path}')

    rgb_arr = np.stack(rgb_feats, axis=0)                          # (T, 1024)
    aud_arr = np.zeros((rgb_arr.shape[0], DIM_AUD), dtype=np.float32)  # (T, 128)
    return rgb_arr, aud_arr


def prepare_tensors(rgb_arr, aud_arr):
    """(T, 1024/128) float → padded+normalized (1, L, D) тензоры + длина."""
    T = min(rgb_arr.shape[0], L)
    rgb = np.zeros((L, DIM_RGB), dtype=np.float32)
    aud = np.zeros((L, DIM_AUD), dtype=np.float32)
    rgb[:T] = (rgb_arr[:T] - norm['rgb_mean']) / norm['rgb_std']
    aud[:T] = (aud_arr[:T] - norm['aud_mean']) / norm['aud_std']
    return (torch.from_numpy(rgb).unsqueeze(0).to(DEVICE),
            torch.from_numpy(aud).unsqueeze(0).to(DEVICE),
            torch.tensor([T], dtype=torch.long))


def predict_genre(video_path, top_k=5, variant='rnn'):
    model = load_model(variant)
    rgb_arr, aud_arr = extract_frame_features(video_path, fps_sample=1, max_frames=L)
    print(f'extracted {rgb_arr.shape[0]} frames')
    rgb, aud, lens = prepare_tensors(rgb_arr, aud_arr)
    with torch.no_grad():
        logits = model(rgb, aud, lens)
        probs  = torch.softmax(logits, dim=-1)[0].cpu().numpy()
    top = np.argsort(probs)[::-1][:top_k]
    print(f'\nPredictions [{variant}]:')
    for i in top:
        print(f'  {GENRES[i]:<14} {probs[i]*100:>5.1f}%')
    return [(GENRES[i], float(probs[i])) for i in top]

In [5]:
# ============================================================
# 6.4 — Скачивание с YouTube (опционально)
# ============================================================
# Тонкости:
# 1) yt-dlp требует JS-рантайм (ставим node через --js-runtimes).
# 2) Для shorts YouTube часто отдаёт av1/opus в webm/mkv — OpenCV без
#    hw-acc такое не декодит. Форсим h264 через vcodec^=avc1 +
#    merge-output-format mp4. Финальный --recode-video mp4 как страховка.
# 3) Итоговый путь берём из --print after_move:filepath, чтобы не
#    гадать по расширению.
import shutil
JS_RUNTIME = None
for name in ('deno', 'node', 'bun'):
    p = shutil.which(name)
    if p:
        JS_RUNTIME = f'{name}:{p}'
        break
print(f'js-runtime: {JS_RUNTIME or "none — yt-dlp будет ругаться"}')


def download_youtube(url, max_seconds=60):
    out_template = str(INFER_DIR / '%(id)s.%(ext)s')
    fmt = ('bv*[vcodec^=avc1][height<=720]+ba/'
           'b[vcodec^=avc1][height<=720]/'
           'b[height<=720]')
    cmd = ['yt-dlp',
           '-f', fmt,
           '--merge-output-format', 'mp4',
           '--recode-video', 'mp4',
           '--no-playlist', '--no-progress',
           '--download-sections', f'*0-{max_seconds}',
           '--print', 'after_move:filepath',
           '-o', out_template, url]
    if JS_RUNTIME:
        cmd += ['--js-runtimes', JS_RUNTIME]
    res = subprocess.run(cmd, check=True, capture_output=True, text=True)
    lines = [ln for ln in res.stdout.strip().splitlines() if ln.strip()]
    if not lines:
        raise RuntimeError(f'yt-dlp ничего не вывел\nstderr:\n{res.stderr}')
    return Path(lines[-1])

js-runtime: node:/usr/bin/node


In [11]:
# ============================================================
# 6.5 — Демо: прогоняем на существующих mp4 из data2/inference/
# ============================================================
demo_dir = Path('../data2/inference')
for mp4 in sorted(demo_dir.glob('*.mp4')):
    print(f'\n=== {mp4.name} ===')
    for variant in ('flatten', 'rnn'):
        try:
            predict_genre(mp4, top_k=3, variant=variant)
        except Exception as e:
            print(f'  [{variant}] error: {e}')


=== -KnFzkY9Kro.mp4 ===
extracted 17 frames

Predictions [flatten]:
  Music           13.4%
  Vehicles        12.8%
  Animation       11.2%
extracted 17 frames

Predictions [rnn]:
  Dance           63.5%
  Gaming           9.0%
  Tech             7.8%

=== 5kNIa6-5eHI.mp4 ===
extracted 42 frames

Predictions [flatten]:
  Gaming          23.1%
  Food            17.6%
  Animation       14.6%
extracted 42 frames

Predictions [rnn]:
  Gaming          51.5%
  Animation       16.5%
  Animals          7.4%

=== 7PVrmGOim90.mp4 ===
extracted 14 frames

Predictions [flatten]:
  Animation       13.0%
  Sports          11.8%
  Dance           10.0%
extracted 14 frames

Predictions [rnn]:
  Sports          30.7%
  Dance           24.8%
  Animation       18.8%

=== dQw4w9WgXcQ.mp4 ===
extracted 60 frames

Predictions [flatten]:
  Music           25.5%
  Performance     13.8%
  Sports          11.0%
extracted 60 frames

Predictions [rnn]:
  Music           25.7%
  Film            18.5%
  Beauty    

In [10]:
# ============================================================
# 6.6 — Или скачать новое по ссылке
# ============================================================
YOUTUBE_URL = 'https://youtube.com/shorts/ZC0cU-9aIWw?si=CMQblzDiqpvgbCQB'
path = download_youtube(YOUTUBE_URL)
print(f'downloaded: {path}')
if path:
    predict_genre(path, top_k=5, variant='rnn')

downloaded: /home/wukker/IdeaProjects/video_classifier/data2/frame_inference/ZC0cU-9aIWw.mp4
extracted 11 frames

Predictions [rnn]:
  Gaming          69.3%
  Music           12.7%
  Animation        5.7%
  Film             2.9%
  Animals          2.5%
